In [3]:
import sys
!{sys.executable} -m pip install opencv-python


Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
     |████████████████████████████████| 72.9 MB 25 kB/s s eta 0:00:01
     |████████████████████████████████| 19.5 MB 98 kB/s s eta 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.5.0 requires daal==2021.4.0, which is not installed.
scipy 1.7.3 requires numpy<1.23.0,>=1.16.5, but you have numpy 2.0.2 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 2.0.2 which is incompatible.


In [4]:
import cv2
print(cv2.__version__)


ModuleNotFoundError: No module named 'cv2'

In [1]:
import cv2
import requests

API_KEY = "W4WFCZURSVGI3VI3"
API_KEY_R = "D0WGJ1JQSV6N24AN"
THINGSPEAK_URL = "https://api.thingspeak.com/update"
detector = cv2.QRCodeDetector()

cap = cv2.VideoCapture(1)

last_data = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    data, bbox, _ = detector.detectAndDecode(frame)

    if data and data != last_data:
        print("QR Code Data:", data)

        try:

            values = dict(item.split(":") for item in data.split(","))

            P = values.get("P")
            D = values.get("D")
            A = values.get("A")
            N = values.get("N")
            M = values.get("M")

            print("P =", P)
            print("D =", D)
            print("A =", A)
            print("N =", N)
            print("M =", M)

            payload = {
                "api_key": API_KEY,
                "field1": P,
                "field2": D,
                "field3": A,
                "field4": N,
                "field5": M,
            }

            response = requests.get(THINGSPEAK_URL, params=payload)

            if response.status_code == 200:
                print("Data sent to ThingSpeak")
                
            payload = {
                "api_key": API_KEY_R,
                "field1": '1'
            }

            response = requests.get(THINGSPEAK_URL, params=payload)

            if response.status_code == 200:
                print("Data sent to ThingSpeak refreshed")

        except Exception as e:
            print("Error parsing QR data:", e)

        last_data = data

    cv2.imshow("QR Scanner", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

QR Code Data: P:1,D:1,A:1,N:1,M:1
P = 1
D = 1
A = 1
N = 1
M = 1
Data sent to ThingSpeak
Data sent to ThingSpeak refreshed
